In [2]:


import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import kagglehub


/kaggle/input/competitions/nfl-big-data-bowl-2027/nfl-big-data-bowl-2027/players.csv
/kaggle/input/competitions/nfl-big-data-bowl-2027/nfl-big-data-bowl-2027/game_tracking_2023.csv
/kaggle/input/competitions/nfl-big-data-bowl-2027/nfl-big-data-bowl-2027/game_tracking_2025.csv
/kaggle/input/competitions/nfl-big-data-bowl-2027/nfl-big-data-bowl-2027/game_tracking_2024.csv
/kaggle/input/competitions/nfl-big-data-bowl-2027/nfl-big-data-bowl-2027/combine_tracking.csv
/kaggle/input/competitions/nfl-big-data-bowl-2027/nfl-big-data-bowl-2027/games.csv
/kaggle/input/competitions/nfl-big-data-bowl-2027/nfl-big-data-bowl-2027/player_career_successes.csv
/kaggle/input/competitions/nfl-big-data-bowl-2027/nfl-big-data-bowl-2027/combine_results.csv
/kaggle/input/competitions/nfl-big-data-bowl-2027/nfl-big-data-bowl-2027/player_play.csv


Research Idea: For wide receivers, do movement features measured during Combine route drills predict their separation on NFL plays—and do they add useful information beyond traditional Combine results?

Data Feasibility checks

In [3]:

import pandas as pd
import numpy as np
import os

BASE_DIR = "/kaggle/input/competitions/nfl-big-data-bowl-2027/nfl-big-data-bowl-2027"

# Load smaller datasets
players_df = pd.read_csv(f"{BASE_DIR}/players.csv")
combine_results_df = pd.read_csv(f"{BASE_DIR}/combine_results.csv")
player_play_df = pd.read_csv(f"{BASE_DIR}/player_play.csv")
games_df = pd.read_csv(f"{BASE_DIR}/games.csv")
career_successes_df = pd.read_csv(
    f"{BASE_DIR}/player_career_successes.csv"
)

print("Players:", players_df.shape)
print("Combine results:", combine_results_df.shape)
print("Player plays:", player_play_df.shape)
print("Games:", games_df.shape)
print("Career successes:", career_successes_df.shape)


Players: (510, 10)
Combine results: (510, 18)
Player plays: (314197, 64)
Games: (1002, 7)
Career successes: (510, 9)


In [4]:

wr_players_df = players_df[
    players_df["nfl_position"].eq("WR")
].copy()

wr_ids = set(wr_players_df["nfl_id"].dropna())

print("Total prospects:", players_df["nfl_id"].nunique())
print("Wide receiver prospects:", len(wr_ids))

display(
    wr_players_df[
        ["nfl_id", "display_name", "draft_year", "nfl_position"]
    ].head(10)
)

print("\nWRs by draft year:")
display(
    wr_players_df.groupby("draft_year")["nfl_id"].nunique()
)


Total prospects: 510
Wide receiver prospects: 106


,nfl_id,display_name,draft_year,nfl_position
7,55884,Jaxon Smith-Njigba,2023,WR
8,55885,Quentin Johnston,2023,WR
9,55887,Jordan Addison,2023,WR
15,55904,Jonathan Mingo,2023,WR
23,55915,Jayden Reed,2023,WR
27,55920,Rashee Rice,2023,WR
32,55928,Marvin Mims,2023,WR
36,55934,Tank Dell,2023,WR
37,55938,Jalin Hyatt,2023,WR
38,55939,Cedric Tillman,2023,WR



WRs by draft year:


draft_year
2023    38
2024    31
2025    37
Name: nfl_id, dtype: int64

In [5]:

combine_tracking_df = pd.read_csv(
    f"{BASE_DIR}/combine_tracking.csv"
)

# Keep only actual player tracking records for WR prospects
combine_wr_df = combine_tracking_df[
    (combine_tracking_df["nfl_id"].isin(wr_ids)) &
    (combine_tracking_df["entity_type"] == "PLAYER")
].copy()

print("Total Combine tracking rows:", len(combine_tracking_df))
print("WR tracking rows:", len(combine_wr_df))

print("\nDrill types:")
display(
    combine_wr_df["drill_type"].value_counts(dropna=False)
)

print("\nDrill names:")
display(
    combine_wr_df["drill_name"].value_counts(dropna=False).head(40)
)


Total Combine tracking rows: 463189
WR tracking rows: 108403

Drill types:


drill_type
SKILL_DRILLS_WR     95400
FORTY_YARD_DASH      8772
THREE_CONE_DRILL     2404
SHORT_SHUTTLE        1827
Name: count, dtype: int64


Drill names:


drill_name
GAUNTLET_DRILL             18081
FORTY_YARD_DASH             8772
POST_CORNER_ROUTE_RIGHT     7875
GO_ROUTE_RIGHT              7583
DAGGER_ROUTE_LEFT           7461
SLOT_SAIL_ROUTE_RIGHT       7332
COMEBACK_ROUTE_RIGHT        6593
OVER_SHOULDER_ADJUST        6543
SLOT_STRIKE_ROUTE_LEFT      6271
CURL_ROUTE_RIGHT            6152
RED_ZONE_FADE_RIGHT         4975
SLANT_ROUTE_LEFT            4280
POST_CORNER_ROUTE_LEFT      4236
SPEED_OUT_ROUTE_LEFT        3792
CIRCUS_ROUTE_LEFT           2551
THREE_CONE_DRILL            2404
SHORT_SHUTTLE               1827
SPEED_OUT_LEFT              1675
Name: count, dtype: int64

In [6]:

# Summarise each individual drill attempt
combine_attempts_df = (
    combine_wr_df
    .groupby(
        ["nfl_id", "draft_year", "event_id", "drill_type", "drill_name"],
        dropna=False
    )
    .agg(
        frame_count=("time", "count"),
        unique_timestamps=("time", "nunique"),
        valid_x=("x", "count"),
        valid_y=("y", "count"),
        valid_speed=("s", "count"),
        valid_acceleration=("a", "count")
    )
    .reset_index()
)

print("Total WR drill attempts:", len(combine_attempts_df))

# Summarise by drill type and drill name
combine_drill_summary_df = (
    combine_attempts_df
    .groupby(["drill_type", "drill_name"], dropna=False)
    .agg(
        unique_players=("nfl_id", "nunique"),
        unique_attempts=("event_id", "nunique"),
        total_frames=("frame_count", "sum"),
        median_frames=("frame_count", "median"),
        mean_frames=("frame_count", "mean")
    )
    .reset_index()
    .sort_values("unique_players", ascending=False)
)

display(combine_drill_summary_df)


Total WR drill attempts: 1753


,drill_type,drill_name,unique_players,unique_attempts,total_frames,median_frames,mean_frames
6,SKILL_DRILLS_WR,GAUNTLET_DRILL,106,198,18081,91.0,91.318182
5,SKILL_DRILLS_WR,DAGGER_ROUTE_LEFT,106,114,7461,64.0,65.447368
12,SKILL_DRILLS_WR,SLANT_ROUTE_LEFT,106,109,4280,43.0,39.266055
4,SKILL_DRILLS_WR,CURL_ROUTE_RIGHT,102,109,6152,56.0,56.440367
14,SKILL_DRILLS_WR,SLOT_STRIKE_ROUTE_LEFT,100,101,6271,61.0,62.089109
8,SKILL_DRILLS_WR,OVER_SHOULDER_ADJUST,99,100,6543,65.0,65.430000
13,SKILL_DRILLS_WR,SLOT_SAIL_ROUTE_RIGHT,97,104,7332,69.0,70.500000
11,SKILL_DRILLS_WR,RED_ZONE_FADE_RIGHT,97,110,4975,44.0,45.227273
7,SKILL_DRILLS_WR,GO_ROUTE_RIGHT,97,104,7583,72.0,72.913462
3,SKILL_DRILLS_WR,COMEBACK_ROUTE_RIGHT,97,109,6593,54.0,60.486239


In [7]:

wr_route_drills_df = combine_drill_summary_df[
    combine_drill_summary_df["drill_type"].eq("SKILL_DRILLS_WR")
].copy()

print("WR-specific drills:")
display(wr_route_drills_df)


WR-specific drills:


,drill_type,drill_name,unique_players,unique_attempts,total_frames,median_frames,mean_frames
6,SKILL_DRILLS_WR,GAUNTLET_DRILL,106,198,18081,91.0,91.318182
5,SKILL_DRILLS_WR,DAGGER_ROUTE_LEFT,106,114,7461,64.0,65.447368
12,SKILL_DRILLS_WR,SLANT_ROUTE_LEFT,106,109,4280,43.0,39.266055
4,SKILL_DRILLS_WR,CURL_ROUTE_RIGHT,102,109,6152,56.0,56.440367
14,SKILL_DRILLS_WR,SLOT_STRIKE_ROUTE_LEFT,100,101,6271,61.0,62.089109
8,SKILL_DRILLS_WR,OVER_SHOULDER_ADJUST,99,100,6543,65.0,65.430000
13,SKILL_DRILLS_WR,SLOT_SAIL_ROUTE_RIGHT,97,104,7332,69.0,70.500000
11,SKILL_DRILLS_WR,RED_ZONE_FADE_RIGHT,97,110,4975,44.0,45.227273
7,SKILL_DRILLS_WR,GO_ROUTE_RIGHT,97,104,7583,72.0,72.913462
3,SKILL_DRILLS_WR,COMEBACK_ROUTE_RIGHT,97,109,6593,54.0,60.486239


In [8]:

MIN_FRAMES = 20

combine_attempts_df["usable_track"] = (
    (combine_attempts_df["frame_count"] >= MIN_FRAMES) &
    (combine_attempts_df["valid_x"] >= MIN_FRAMES) &
    (combine_attempts_df["valid_y"] >= MIN_FRAMES) &
    (combine_attempts_df["valid_speed"] >= MIN_FRAMES) &
    (combine_attempts_df["valid_acceleration"] >= MIN_FRAMES)
)

combine_quality_summary_df = (
    combine_attempts_df
    .groupby(["drill_type", "drill_name"], dropna=False)
    .agg(
        total_attempts=("event_id", "nunique"),
        players=("nfl_id", "nunique"),
        usable_attempts=("usable_track", "sum"),
        median_frames=("frame_count", "median")
    )
    .reset_index()
)

combine_quality_summary_df["usable_attempt_percentage"] = (
    combine_quality_summary_df["usable_attempts"] /
    combine_quality_summary_df["total_attempts"] * 100
)

display(
    combine_quality_summary_df.sort_values(
        "players", ascending=False
    )
)


,drill_type,drill_name,total_attempts,players,usable_attempts,median_frames,usable_attempt_percentage
6,SKILL_DRILLS_WR,GAUNTLET_DRILL,198,106,198,91.0,100.0
5,SKILL_DRILLS_WR,DAGGER_ROUTE_LEFT,114,106,114,64.0,100.0
12,SKILL_DRILLS_WR,SLANT_ROUTE_LEFT,109,106,109,43.0,100.0
4,SKILL_DRILLS_WR,CURL_ROUTE_RIGHT,109,102,109,56.0,100.0
14,SKILL_DRILLS_WR,SLOT_STRIKE_ROUTE_LEFT,101,100,101,61.0,100.0
8,SKILL_DRILLS_WR,OVER_SHOULDER_ADJUST,100,99,100,65.0,100.0
13,SKILL_DRILLS_WR,SLOT_SAIL_ROUTE_RIGHT,104,97,104,69.0,100.0
11,SKILL_DRILLS_WR,RED_ZONE_FADE_RIGHT,110,97,110,44.0,100.0
7,SKILL_DRILLS_WR,GO_ROUTE_RIGHT,104,97,104,72.0,100.0
3,SKILL_DRILLS_WR,COMEBACK_ROUTE_RIGHT,109,97,109,54.0,100.0


In [9]:

combine_wr_results_df = combine_results_df[
    (combine_results_df["nfl_id"].isin(wr_ids)) &
    (combine_results_df["combine_position"] == "WR")
].copy()

traditional_combine_features = [
    "combine_height",
    "combine_weight",
    "forty",
    "ten_yd_split",
    "vertical",
    "broad_jump",
    "three_cone",
    "short_shuttle",
    "bench_reps",
    "ngs_athleticism_score",
    "ngs_college_production_score",
    "ngs_final_score"
]

combine_results_availability_df = (
    combine_wr_results_df[traditional_combine_features]
    .notna()
    .sum()
    .reset_index()
)

combine_results_availability_df.columns = [
    "feature", "available_players"
]

combine_results_availability_df["missing_players"] = (
    len(combine_wr_results_df) -
    combine_results_availability_df["available_players"]
)

display(combine_results_availability_df)


,feature,available_players,missing_players
0,combine_height,106,0
1,combine_weight,106,0
2,forty,96,10
3,ten_yd_split,96,10
4,vertical,95,11
5,broad_jump,94,12
6,three_cone,32,74
7,short_shuttle,38,68
8,bench_reps,28,78
9,ngs_athleticism_score,106,0


In [10]:

combine_wr_results_df = combine_results_df[
    (combine_results_df["nfl_id"].isin(wr_ids)) &
    (combine_results_df["combine_position"] == "WR")
].copy()

traditional_combine_features = [
    "combine_height",
    "combine_weight",
    "forty",
    "ten_yd_split",
    "vertical",
    "broad_jump",
    "three_cone",
    "short_shuttle",
    "bench_reps",
    "ngs_athleticism_score",
    "ngs_college_production_score",
    "ngs_final_score"
]

combine_results_availability_df = (
    combine_wr_results_df[traditional_combine_features]
    .notna()
    .sum()
    .reset_index()
)

combine_results_availability_df.columns = [
    "feature", "available_players"
]

combine_results_availability_df["missing_players"] = (
    len(combine_wr_results_df) -
    combine_results_availability_df["available_players"]
)

display(combine_results_availability_df)


,feature,available_players,missing_players
0,combine_height,106,0
1,combine_weight,106,0
2,forty,96,10
3,ten_yd_split,96,10
4,vertical,95,11
5,broad_jump,94,12
6,three_cone,32,74
7,short_shuttle,38,68
8,bench_reps,28,78
9,ngs_athleticism_score,106,0


In [11]:

# Filter to targeted WR prospects
wr_targets_df = player_play_df[
    (player_play_df["nfl_id"].isin(wr_ids)) &
    (player_play_df["target"] == True)
].copy()

print("Total targeted WR play records:", len(wr_targets_df))

print(
    "Unique targeted WRs:",
    wr_targets_df["nfl_id"].nunique()
)

print(
    "Unique target plays:",
    wr_targets_df[["game_id", "play_id"]].drop_duplicates().shape[0]
)

# Check separation availability
print("\nSeparation availability:")
print(wr_targets_df["separation_at_pass_forward"].describe())

print(
    "\nMissing separation:",
    wr_targets_df["separation_at_pass_forward"].isna().sum()
)

# Keep observations with valid separation
wr_targets_with_separation_df = wr_targets_df[
    wr_targets_df["separation_at_pass_forward"].notna()
].copy()

print(
    "\nTargets with valid separation:",
    len(wr_targets_with_separation_df)
)

print(
    "WRs with valid separation:",
    wr_targets_with_separation_df["nfl_id"].nunique()
)


Total targeted WR play records: 8093
Unique targeted WRs: 104
Unique target plays: 8093

Separation availability:
count    8093.000000
mean        3.588434
std         2.350974
min         0.224722
25%         1.623392
50%         3.116360
75%         5.035603
max        17.716084
Name: separation_at_pass_forward, dtype: float64

Missing separation: 0

Targets with valid separation: 8093
WRs with valid separation: 104


In [12]:

# Add season from game metadata
wr_targets_with_separation_df = (
    wr_targets_with_separation_df
    .merge(
        games_df[["game_id", "season", "season_type"]],
        on="game_id",
        how="left",
        validate="many_to_one"
    )
)

# Summarise targets by season
nfl_season_summary_df = (
    wr_targets_with_separation_df
    .groupby("season")
    .agg(
        target_observations=("play_id", "size"),
        unique_players=("nfl_id", "nunique"),
        unique_games=("game_id", "nunique"),
        mean_separation=("separation_at_pass_forward", "mean"),
        median_separation=("separation_at_pass_forward", "median")
    )
    .reset_index()
)

display(nfl_season_summary_df)


,season,target_observations,unique_players,unique_games,mean_separation,median_separation
0,2023,1730,38,259,3.694064,3.186173
1,2024,2770,64,311,3.581646,3.117138
2,2025,3593,91,315,3.542808,3.065453


In [13]:

wr_target_counts_df = (
    wr_targets_with_separation_df
    .groupby("nfl_id")
    .agg(
        target_count=("play_id", "size"),
        unique_games=("game_id", "nunique"),
        mean_separation=("separation_at_pass_forward", "mean"),
        median_separation=("separation_at_pass_forward", "median")
    )
    .reset_index()
    .sort_values("target_count", ascending=False)
)

display(wr_target_counts_df.describe())

print("\nReceivers with at least:")
for threshold in [5, 10, 20, 50, 100]:
    count = (wr_target_counts_df["target_count"] >= threshold).sum()
    print(f"{threshold} targets: {count} WRs")


,nfl_id,target_count,unique_games,mean_separation,median_separation
count,104.000000,104.000000,104.000000,104.000000,104.000000
mean,57279.471154,77.817308,19.250000,3.493899,3.036254
std,1039.569375,92.956902,15.898189,0.908758,1.068939
min,55884.000000,1.000000,1.000000,1.346477,0.796016
25%,56178.750000,14.000000,5.000000,2.946637,2.288287
50%,57232.500000,40.500000,15.500000,3.355058,2.952086
75%,58313.500000,105.250000,32.000000,4.026915,3.599795
max,59278.000000,512.000000,58.000000,7.306593,7.220997



Receivers with at least:
5 targets: 98 WRs
10 targets: 84 WRs
20 targets: 71 WRs
50 targets: 45 WRs
100 targets: 28 WRs


In [14]:

# WRs with Combine tracking
combine_wr_ids = set(
    combine_wr_df["nfl_id"].dropna().unique()
)

# WRs with NFL targets and valid separation
nfl_wr_ids = set(
    wr_targets_with_separation_df["nfl_id"].dropna().unique()
)

# Players appearing in both datasets
combine_nfl_overlap_ids = combine_wr_ids.intersection(nfl_wr_ids)

print("WRs with Combine tracking:", len(combine_wr_ids))
print("WRs with NFL separation observations:", len(nfl_wr_ids))
print("WRs appearing in both:", len(combine_nfl_overlap_ids))

print(
    "Combine WRs with NFL separation data:",
    round(
        len(combine_nfl_overlap_ids) /
        max(len(combine_wr_ids), 1) * 100,
        2
    ),
    "%"
)


WRs with Combine tracking: 106
WRs with NFL separation observations: 104
WRs appearing in both: 104
Combine WRs with NFL separation data: 98.11 %


In [15]:

# Only retain usable Combine attempts
usable_combine_attempts_df = combine_attempts_df[
    combine_attempts_df["usable_track"]
].copy()

# Count unique WRs with usable Combine tracks and NFL separation
drill_nfl_overlap_df = (
    usable_combine_attempts_df[
        usable_combine_attempts_df["nfl_id"].isin(nfl_wr_ids)
    ]
    .groupby(["drill_type", "drill_name"])
    .agg(
        combine_nfl_players=("nfl_id", "nunique"),
        usable_attempts=("event_id", "nunique")
    )
    .reset_index()
    .sort_values("combine_nfl_players", ascending=False)
)

display(drill_nfl_overlap_df)


,drill_type,drill_name,combine_nfl_players,usable_attempts
6,SKILL_DRILLS_WR,GAUNTLET_DRILL,104,194
5,SKILL_DRILLS_WR,DAGGER_ROUTE_LEFT,104,111
12,SKILL_DRILLS_WR,SLANT_ROUTE_LEFT,104,107
4,SKILL_DRILLS_WR,CURL_ROUTE_RIGHT,100,107
14,SKILL_DRILLS_WR,SLOT_STRIKE_ROUTE_LEFT,99,100
8,SKILL_DRILLS_WR,OVER_SHOULDER_ADJUST,97,98
13,SKILL_DRILLS_WR,SLOT_SAIL_ROUTE_RIGHT,95,102
11,SKILL_DRILLS_WR,RED_ZONE_FADE_RIGHT,95,108
7,SKILL_DRILLS_WR,GO_ROUTE_RIGHT,95,102
3,SKILL_DRILLS_WR,COMEBACK_ROUTE_RIGHT,95,107


In [16]:

# Players with usable Combine tracks
usable_combine_wr_ids = set(
    usable_combine_attempts_df["nfl_id"].unique()
)

# Players with traditional Combine results
traditional_combine_wr_ids = set(
    combine_wr_results_df["nfl_id"].unique()
)

# Players with NFL separation observations
nfl_separation_wr_ids = set(
    wr_targets_with_separation_df["nfl_id"].unique()
)

# Complete player intersection
complete_research_ids = (
    usable_combine_wr_ids
    & traditional_combine_wr_ids
    & nfl_separation_wr_ids
)

print("WRs with usable Combine tracking:", len(usable_combine_wr_ids))
print("WRs with traditional Combine results:", len(traditional_combine_wr_ids))
print("WRs with NFL separation:", len(nfl_separation_wr_ids))

print("\nFinal overlapping WR sample:", len(complete_research_ids))


WRs with usable Combine tracking: 106
WRs with traditional Combine results: 106
WRs with NFL separation: 104

Final overlapping WR sample: 104


In [17]:

drill_final_sample_df = (
    usable_combine_attempts_df[
        usable_combine_attempts_df["nfl_id"].isin(
            traditional_combine_wr_ids & nfl_separation_wr_ids
        )
    ]
    .groupby(["drill_type", "drill_name"])
    .agg(
        final_players=("nfl_id", "nunique"),
        usable_attempts=("event_id", "nunique")
    )
    .reset_index()
    .sort_values("final_players", ascending=False)
)

display(drill_final_sample_df)


,drill_type,drill_name,final_players,usable_attempts
6,SKILL_DRILLS_WR,GAUNTLET_DRILL,104,194
5,SKILL_DRILLS_WR,DAGGER_ROUTE_LEFT,104,111
12,SKILL_DRILLS_WR,SLANT_ROUTE_LEFT,104,107
4,SKILL_DRILLS_WR,CURL_ROUTE_RIGHT,100,107
14,SKILL_DRILLS_WR,SLOT_STRIKE_ROUTE_LEFT,99,100
8,SKILL_DRILLS_WR,OVER_SHOULDER_ADJUST,97,98
13,SKILL_DRILLS_WR,SLOT_SAIL_ROUTE_RIGHT,95,102
11,SKILL_DRILLS_WR,RED_ZONE_FADE_RIGHT,95,108
7,SKILL_DRILLS_WR,GO_ROUTE_RIGHT,95,102
3,SKILL_DRILLS_WR,COMEBACK_ROUTE_RIGHT,95,107


In [18]:

# Count valid separation observations for each WR
wr_separation_counts_df = (
    wr_targets_with_separation_df
    .groupby("nfl_id")
    .agg(
        valid_targets=("separation_at_pass_forward", "count"),
        mean_separation=("separation_at_pass_forward", "mean"),
        median_separation=("separation_at_pass_forward", "median"),
        std_separation=("separation_at_pass_forward", "std")
    )
    .reset_index()
)

# Add player names
wr_separation_counts_df = wr_separation_counts_df.merge(
    players_df[["nfl_id", "display_name", "draft_year"]],
    on="nfl_id",
    how="left"
)

for threshold in [5, 10, 20, 30, 50]:
    count = (
        wr_separation_counts_df["valid_targets"] >= threshold
    ).sum()

    print(f"WRs with at least {threshold} valid targets: {count}")

display(
    wr_separation_counts_df.sort_values(
        "valid_targets", ascending=False
    ).head(20)
)


WRs with at least 5 valid targets: 98
WRs with at least 10 valid targets: 84
WRs with at least 20 valid targets: 71
WRs with at least 30 valid targets: 60
WRs with at least 50 valid targets: 45


,nfl_id,valid_targets,mean_separation,median_separation,std_separation,display_name,draft_year
15,56042,512,3.389956,3.000633,2.059604,Puka Nacua,2023
0,55884,429,3.785107,3.365828,2.268008,Jaxon Smith-Njigba,2023
2,55887,303,3.491896,3.235568,2.094182,Jordan Addison,2023
5,55920,269,4.262172,4.074162,2.262633,Rashee Rice,2023
1,55885,268,3.275960,2.765866,2.222101,Quentin Johnston,2023
10,55959,258,3.278195,2.484475,2.370416,Michael Wilson,2023
43,57155,238,3.575717,3.026286,2.372673,Ladd McConkey,2024
39,57144,236,3.226008,2.770883,1.983102,Brian Thomas,2024
21,56075,233,4.480586,4.187027,2.798300,Demario Douglas,2023
11,55965,231,3.693697,3.240772,2.406559,Tre Tucker,2023


In [19]:

feasibility_summary_df = pd.DataFrame({
    "Stage": [
        "All NFL prospects",
        "Wide receiver prospects",
        "WRs with Combine tracking",
        "WRs with usable Combine tracks",
        "WRs with traditional Combine results",
        "WRs with NFL separation observations",
        "WRs with all three data sources"
    ],
    "Unique players": [
        players_df["nfl_id"].nunique(),
        len(wr_ids),
        len(combine_wr_ids),
        len(usable_combine_wr_ids),
        len(traditional_combine_wr_ids),
        len(nfl_separation_wr_ids),
        len(complete_research_ids)
    ]
})

display(feasibility_summary_df)

print("\nFinal drill-level feasibility:")
display(drill_final_sample_df)


,Stage,Unique players
0,All NFL prospects,510
1,Wide receiver prospects,106
2,WRs with Combine tracking,106
3,WRs with usable Combine tracks,106
4,WRs with traditional Combine results,106
5,WRs with NFL separation observations,104
6,WRs with all three data sources,104



Final drill-level feasibility:


,drill_type,drill_name,final_players,usable_attempts
6,SKILL_DRILLS_WR,GAUNTLET_DRILL,104,194
5,SKILL_DRILLS_WR,DAGGER_ROUTE_LEFT,104,111
12,SKILL_DRILLS_WR,SLANT_ROUTE_LEFT,104,107
4,SKILL_DRILLS_WR,CURL_ROUTE_RIGHT,100,107
14,SKILL_DRILLS_WR,SLOT_STRIKE_ROUTE_LEFT,99,100
8,SKILL_DRILLS_WR,OVER_SHOULDER_ADJUST,97,98
13,SKILL_DRILLS_WR,SLOT_SAIL_ROUTE_RIGHT,95,102
11,SKILL_DRILLS_WR,RED_ZONE_FADE_RIGHT,95,108
7,SKILL_DRILLS_WR,GO_ROUTE_RIGHT,95,102
3,SKILL_DRILLS_WR,COMEBACK_ROUTE_RIGHT,95,107


In [20]:

for year in [2023, 2024, 2025]:

    tracking_path = f"{BASE_DIR}/game_tracking_{year}.csv"

    tracking_sample_df = pd.read_csv(
        tracking_path,
        nrows=5
    )

    print(f"\nNFL TRACKING {year}")
    print("Columns:", tracking_sample_df.columns.tolist())

    # Count rows for the WR cohort using chunks
    wr_tracking_rows = 0
    wr_tracking_players = set()

    for chunk in pd.read_csv(
        tracking_path,
        chunksize=500_000,
        low_memory=False
    ):
        wr_chunk = chunk[
            chunk["nfl_id"].isin(wr_ids)
        ]

        wr_tracking_rows += len(wr_chunk)

        wr_tracking_players.update(
            wr_chunk["nfl_id"].dropna().unique()
        )

    print("WR tracking rows:", wr_tracking_rows)
    print("Unique WRs tracked:", len(wr_tracking_players))



NFL TRACKING 2023
Columns: ['game_id', 'play_id', 'nfl_id', 'time', 'x', 'y', 's', 'a', 'dis', 'o', 'dir', 'event']
WR tracking rows: 719353
Unique WRs tracked: 38

NFL TRACKING 2024
Columns: ['game_id', 'play_id', 'nfl_id', 'time', 'x', 'y', 's', 'a', 'dis', 'o', 'dir', 'event']
WR tracking rows: 1158601
Unique WRs tracked: 65

NFL TRACKING 2025
Columns: ['game_id', 'play_id', 'nfl_id', 'time', 'x', 'y', 's', 'a', 'dis', 'o', 'dir', 'event']
WR tracking rows: 1475631
Unique WRs tracked: 93


Feature extraction

In [21]:

# Select the five route drills
selected_route_drills = [
    "DAGGER_ROUTE_LEFT",
    "SLANT_ROUTE_LEFT",
    "CURL_ROUTE_RIGHT",
    "GO_ROUTE_RIGHT",
    "COMEBACK_ROUTE_RIGHT"
]

combine_route_tracking_df = combine_wr_df[
    combine_wr_df["drill_name"].isin(selected_route_drills)
].copy()

# Convert timestamps
combine_route_tracking_df["timestamp"] = pd.to_datetime(
    combine_route_tracking_df["time"],
    errors="coerce"
)

# Sort tracking frames chronologically
combine_route_tracking_df = combine_route_tracking_df.sort_values(
    ["nfl_id", "event_id", "timestamp"]
)


def extract_movement_features(group):

    group = group.sort_values("timestamp")

    # Extract valid coordinates
    coordinates = group[["x", "y"]].dropna()

    # Calculate movement distance from consecutive coordinates
    if len(coordinates) >= 2:

        dx = np.diff(coordinates["x"].to_numpy())
        dy = np.diff(coordinates["y"].to_numpy())

        frame_distances = np.sqrt(dx**2 + dy**2)

        total_distance = frame_distances.sum()

        net_displacement = np.sqrt(
            (coordinates["x"].iloc[-1] - coordinates["x"].iloc[0])**2 +
            (coordinates["y"].iloc[-1] - coordinates["y"].iloc[0])**2
        )

        straightness_ratio = (
            net_displacement / total_distance
            if total_distance > 0 else np.nan
        )

        lateral_range = (
            coordinates["y"].max() - coordinates["y"].min()
        )

        longitudinal_range = (
            coordinates["x"].max() - coordinates["x"].min()
        )

    else:
        total_distance = np.nan
        net_displacement = np.nan
        straightness_ratio = np.nan
        lateral_range = np.nan
        longitudinal_range = np.nan

    # Speed measurements
    speed = group["s"].dropna()

    # Acceleration measurements
    acceleration = group["a"].dropna()

    # Direction changes using circular angle differences
    direction = group["dir"].dropna().to_numpy()

    if len(direction) >= 2:

        direction_radians = np.deg2rad(direction)

        direction_changes = np.diff(direction_radians)

        # Correct for the 0/360-degree boundary
        direction_changes = (
            (direction_changes + np.pi) % (2 * np.pi)
        ) - np.pi

        total_direction_change = np.abs(
            np.rad2deg(direction_changes)
        ).sum()

    else:
        total_direction_change = np.nan

    # Drill duration
    duration_seconds = (
        group["timestamp"].max() - group["timestamp"].min()
    ).total_seconds()

    return pd.Series({

        "frame_count": len(group),

        "duration_seconds": duration_seconds,

        "mean_speed": speed.mean(),
        "max_speed": speed.max(),
        "speed_std": speed.std(),

        "max_abs_acceleration": acceleration.abs().max(),
        "mean_abs_acceleration": acceleration.abs().mean(),
        "acceleration_std": acceleration.std(),

        "total_distance_yards": total_distance,
        "net_displacement_yards": net_displacement,
        "straightness_ratio": straightness_ratio,

        "lateral_range_yards": lateral_range,
        "longitudinal_range_yards": longitudinal_range,

        "total_direction_change_deg": total_direction_change

    })


# Calculate features for each individual attempt
attempt_feature_rows = []

attempt_group_columns = [
    "nfl_id",
    "draft_year",
    "event_id",
    "attempt",
    "drill_name"
]

for group_values, group in combine_route_tracking_df.groupby(
    attempt_group_columns,
    dropna=False
):

    features = extract_movement_features(group)

    row = dict(zip(attempt_group_columns, group_values))
    row.update(features.to_dict())

    attempt_feature_rows.append(row)


combine_movement_attempts_df = pd.DataFrame(
    attempt_feature_rows
)

print("Movement feature rows:", len(combine_movement_attempts_df))

display(combine_movement_attempts_df.head())


Movement feature rows: 545


,nfl_id,draft_year,event_id,attempt,drill_name,frame_count,duration_seconds,mean_speed,max_speed,speed_std,max_abs_acceleration,mean_abs_acceleration,acceleration_std,total_distance_yards,net_displacement_yards,straightness_ratio,lateral_range_yards,longitudinal_range_yards,total_direction_change_deg
0,55884,2023,6414e684e3d8e72dcdcb4e0f-2023,1,SLANT_ROUTE_LEFT,50.0,4.9,5.713600,8.756667,2.964343,4.587307,2.681309,1.138042,28.209887,27.246305,0.965842,17.400000,20.966667,128.400293
1,55884,2023,6414e684e3d8e72dcdcb4e76-2023,1,DAGGER_ROUTE_LEFT,64.0,6.3,6.937708,8.673333,2.156711,7.299085,3.784838,1.767140,44.093647,38.701881,0.877720,24.033333,30.466667,144.449155
2,55885,2023,6414e684e3d8e72dcdcb4df1-2023,1,SLANT_ROUTE_LEFT,44.0,4.3,5.463712,8.683333,2.724678,4.443848,2.932014,1.278315,23.699995,22.990046,0.970044,8.900000,21.266667,137.646426
3,55885,2023,6414e684e3d8e72dcdcb4e59-2023,1,DAGGER_ROUTE_LEFT,61.0,6.0,6.515355,8.586667,1.965080,6.480042,3.632070,1.674773,39.422193,34.916440,0.885705,17.133333,30.700000,124.817290
4,55885,2023,6414e684e3d8e72dcdcb4f1d-2023,1,CURL_ROUTE_RIGHT,53.0,5.2,3.707170,7.060000,2.032643,7.702788,4.439652,1.987934,19.509137,15.075993,0.772766,3.000000,15.033333,300.485623


In [22]:

movement_feature_columns = [
    "duration_seconds",
    "mean_speed",
    "max_speed",
    "speed_std",
    "max_abs_acceleration",
    "mean_abs_acceleration",
    "acceleration_std",
    "total_distance_yards",
    "net_displacement_yards",
    "straightness_ratio",
    "lateral_range_yards",
    "longitudinal_range_yards",
    "total_direction_change_deg"
]

player_drill_features_df = (
    combine_movement_attempts_df
    .groupby(
        ["nfl_id", "draft_year", "drill_name"],
        dropna=False
    )
    .agg(
        attempt_count=("event_id", "nunique"),
        **{
            feature: (feature, "median")
            for feature in movement_feature_columns
        }
    )
    .reset_index()
)

print("Player-drill observations:", len(player_drill_features_df))

display(
    player_drill_features_df.sort_values(
        ["drill_name", "nfl_id"]
    ).head(20)
)


Player-drill observations: 508


,nfl_id,draft_year,drill_name,attempt_count,duration_seconds,mean_speed,max_speed,speed_std,max_abs_acceleration,mean_abs_acceleration,acceleration_std,total_distance_yards,net_displacement_yards,straightness_ratio,lateral_range_yards,longitudinal_range_yards,total_direction_change_deg
2,55885,2023,COMEBACK_ROUTE_RIGHT,1,4.9,4.954133,8.233333,2.038474,7.485145,3.876657,2.052243,24.719808,19.441651,0.786481,8.400000,17.733333,148.828048
9,55904,2023,COMEBACK_ROUTE_RIGHT,1,4.8,4.889524,8.530000,2.185145,7.833979,4.313716,1.996525,23.938840,18.149992,0.758182,6.733333,18.066667,160.497611
14,55915,2023,COMEBACK_ROUTE_RIGHT,1,5.3,5.041790,8.446667,2.072801,8.397334,4.386059,2.195841,27.263691,18.144727,0.665527,9.300000,18.333333,167.953843
19,55920,2023,COMEBACK_ROUTE_RIGHT,1,4.7,5.289653,8.326667,2.064313,8.494507,4.016199,2.299539,25.294115,18.776167,0.742314,7.900000,18.400000,137.480721
24,55928,2023,COMEBACK_ROUTE_RIGHT,1,5.1,4.877821,8.533333,2.228995,8.399196,4.427739,2.280026,25.428839,16.845705,0.662465,7.400000,18.000000,185.391341
29,55934,2023,COMEBACK_ROUTE_RIGHT,1,5.2,4.997547,8.290000,2.198996,8.449766,4.208471,2.277719,26.547248,17.411937,0.655885,8.566667,18.400000,150.075153
38,55939,2023,COMEBACK_ROUTE_RIGHT,1,4.8,4.825986,7.940000,2.024272,7.811420,3.945270,2.138729,23.616935,17.123084,0.725034,6.600000,17.533333,137.066564
43,55959,2023,COMEBACK_ROUTE_RIGHT,1,5.0,5.489673,8.433333,2.045675,7.878390,4.181487,2.157167,27.909282,19.577907,0.701484,9.533333,19.400000,137.594172
48,55965,2023,COMEBACK_ROUTE_RIGHT,1,5.0,5.528170,8.360000,2.085665,8.402006,4.027710,2.291602,27.954711,19.084956,0.682710,9.800000,18.733333,128.626794
53,55990,2023,COMEBACK_ROUTE_RIGHT,1,5.2,4.755283,8.256667,2.236767,7.823414,3.748100,2.211760,25.135648,17.923603,0.713075,8.800000,17.166667,325.860653


In [23]:

# Count players available for each drill
movement_coverage_df = (
    player_drill_features_df
    .groupby("drill_name")
    .agg(
        unique_players=("nfl_id", "nunique"),
        total_attempts=("attempt_count", "sum"),
        median_attempts=("attempt_count", "median")
    )
    .reset_index()
    .sort_values("unique_players", ascending=False)
)

display(movement_coverage_df)

# Check missing values in the movement features
movement_missing_df = (
    player_drill_features_df[movement_feature_columns]
    .isna()
    .sum()
    .reset_index()
)

movement_missing_df.columns = [
    "feature", "missing_values"
]

display(movement_missing_df)

# Inspect descriptive statistics
display(
    player_drill_features_df[
        movement_feature_columns
    ].describe().round(3)
)



,drill_name,unique_players,total_attempts,median_attempts
2,DAGGER_ROUTE_LEFT,106,114,1.0
4,SLANT_ROUTE_LEFT,106,109,1.0
1,CURL_ROUTE_RIGHT,102,109,1.0
0,COMEBACK_ROUTE_RIGHT,97,109,1.0
3,GO_ROUTE_RIGHT,97,104,1.0


,feature,missing_values
0,duration_seconds,0
1,mean_speed,0
2,max_speed,0
3,speed_std,0
4,max_abs_acceleration,0
5,mean_abs_acceleration,0
6,acceleration_std,0
7,total_distance_yards,0
8,net_displacement_yards,0
9,straightness_ratio,0


,duration_seconds,mean_speed,max_speed,speed_std,max_abs_acceleration,mean_abs_acceleration,acceleration_std,total_distance_yards,net_displacement_yards,straightness_ratio,lateral_range_yards,longitudinal_range_yards,total_direction_change_deg
count,508.000,508.000,508.000,508.000,508.000,508.000,508.000,508.000,508.000,508.000,508.000,508.000,508.000
mean,5.757,5.682,8.366,2.236,6.603,3.506,1.682,33.786,29.688,0.859,9.726,27.868,165.138
std,1.443,1.317,1.052,0.371,1.440,0.874,0.395,13.553,14.732,0.121,5.376,14.699,98.193
min,2.000,3.090,5.816,1.392,3.835,1.651,0.778,11.335,7.005,0.264,0.564,9.043,13.832
25%,4.900,4.680,7.661,1.954,5.156,2.979,1.389,23.904,17.744,0.772,6.033,17.517,83.123
50%,5.900,5.314,8.218,2.160,6.807,3.739,1.622,28.237,23.432,0.886,8.328,21.000,148.709
75%,6.800,6.578,8.811,2.521,7.809,4.154,1.998,42.150,36.489,0.964,11.371,30.950,246.911
max,14.200,8.536,10.977,3.568,9.839,5.229,2.680,59.608,59.108,1.000,29.267,58.567,455.411
